In [2]:
import numpy as np
import pandas as pd

# For x_t > 0: linear x_{t+1}/x_t = r; nonlinear x_{t+1}/x_t = r(1-x_t).
# In the nonlinear rule, the proportional factor depends on the current state.
def logistic_curve(x, r):
    return r * x * (1 - x)

r_value = 2.5
x_values = np.linspace(0, 1, 400)

df_rule = pd.DataFrame({
    'x_t': x_values,
    'x_next': logistic_curve(x_values, r_value),
    'r': r_value
})

df_rule


,x_t,x_next,r
0,0.000000,0.000000,2.5
1,0.002506,0.006250,2.5
2,0.005013,0.012469,2.5
3,0.007519,0.018656,2.5
4,0.010025,0.024811,2.5
...,...,...,...
395,0.989975,0.024811,2.5
396,0.992481,0.018656,2.5
397,0.994987,0.012469,2.5
398,0.997494,0.006250,2.5


In [3]:
import altair as alt

# Include the exact states where x_{t+1} = x_t, and the maximum.
x_star = 1 - 1 / r_value
x_plot = np.unique(np.append(df_rule['x_t'].to_numpy(), [0, x_star, 0.5]))
y_plot = logistic_curve(x_plot, r_value)
unchanged = np.isclose(y_plot, x_plot, rtol=0, atol=1e-12)

# Color each point by comparing the next state with the current state.
df_plot = pd.DataFrame({
    'x_t': x_plot,
    'x_next': y_plot,
    'change': y_plot - x_plot,
    'State': np.where(unchanged, 'Unchanged',
                     np.where(y_plot > x_plot, 'Increases', 'Decreases'))
})

tooltips = [
    alt.Tooltip('x_t:Q', title='Current state (xₜ)', format='.4f'),
    alt.Tooltip('x_next:Q', title='Next state (xₜ₊₁)', format='.4f'),
    alt.Tooltip('change:Q', title='Change (xₜ₊₁ − xₜ)', format='+.4f'),
    alt.Tooltip('State:N', title='State')
]

points = alt.Chart(df_plot).mark_circle(opacity=0.8).encode(
    x=alt.X('x_t:Q', title='Current state xₜ', scale=alt.Scale(domain=[0, 1])),
    y=alt.Y('x_next:Q', title='Next state xₜ₊₁', scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('State:N',
        scale=alt.Scale(domain=['Increases', 'Decreases', 'Unchanged'],
                        range=['#2ca02c', '#d62728', '#1f77b4']),
        legend=alt.Legend(title='Compared with xₜ', orient='top')),
    size=alt.condition(alt.datum.State == 'Unchanged', alt.value(120), alt.value(35)),
    tooltip=tooltips
)

# Identity line: points above increase; points below decrease.
identity = alt.Chart(pd.DataFrame({'x': [0, 1], 'y': [0, 1]})).mark_line(
    color='gray', strokeDash=[6, 4], opacity=0.7
).encode(x='x:Q', y='y:Q')

# An orange outline marks the maximum without hiding its state color.
maximum = alt.Chart(df_plot[df_plot['x_t'] == 0.5]).mark_point(
    color='orange', filled=False, size=200, strokeWidth=3
).encode(x='x_t:Q', y='x_next:Q', tooltip=tooltips)

chart = (identity + points + maximum).properties(
    width=600, height=500,
    title=alt.TitleParams(
        text=f'Logistic Map relevant points (r = {r_value})',
        subtitle=['Hover over a point to read xₜ and xₜ₊₁.',
                  'Dashed line: xₜ₊₁ = xₜ. Orange ring: maximum next state.']
    )
)

chart


<Altair chart>